# Notebook_A (Sinh viên A): dữ liệu, EDA, làm sạch, SQL, hình RQ1

**Đề tài**: Chất lượng nước sông 3 ngày tại 30 trạm USGS (NWIS)

**Khung SDG**: SDG 6 và 14: "water governance", "river restoration"; tiêu đề, keyword và abstract của bài dùng đúng cụm từ này.

**Vai trò**: Sinh viên A làm Bước 0 đến 3 và phần hình RQ1 của Bước 5, bàn giao `data/processed/feat.parquet` cùng `manifest.json` cho Sinh viên B (Notebook_B). Chạy tuần tự trong VS Code với extension Python và Jupyter, interpreter `.venv`. Mỗi bước có ô hướng dẫn *làm gì, vì sao, đọc thế nào* rồi đến ô code; mọi bảng lưu vào `report/` với tên `table_*.csv`, mọi hình `fig_*.png`.

**Quy tắc**: seed 42; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân; báo cáo slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45) theo mẫu ở ô cuối.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [3]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


## Vì sao sáu bước và mỗi bước giúp gì cho bài

| Bước | Mục đích | Lợi ích cho bài | Bằng chứng, câu hỏi |
|---|---|---|---|
| 1 Nạp, hiểu dữ liệu | biết quy mô, thời gian, đơn vị, mục tiêu | bảng mô tả dữ liệu, phạm vi hợp lệ | table_describe, profile; nền mọi RQ |
| 2 Làm sạch, ghép | dữ liệu tin được, có nguồn thứ hai | nhật ký làm sạch trả lời phản biện; đặc trưng ngoại sinh | cleaning_log, tỷ lệ khớp; RQ1, RQ2 |
| 3 SQL, EDA | trả lời câu hỏi mô tả bằng số có kiểm định; đặc trưng đúng thời điểm | bảng RQ1 có p-value; không rò rỉ; điểm SQL | table_q*, rq1a-c; RQ1 |
| 4 Chia dữ liệu | đánh giá công bằng | mốc naive cho biết cải thiện thật | table_features; RQ2 |
| 5 Trực quan hoá | thấy cấu trúc và bất thường; truyền đạt | hình có caption kết luận | fig_rq1_*; RQ1 |
| 6 Mô hình, RQ3 | so năm mô hình công bằng; đóng góp chính | bảng có độ lệch, tinh chỉnh có căn cứ, SHAP | table_rq2, rq3; RQ2, RQ3 |

**Vì sao phải chọn cột từ nhiều cột**: cột thừa làm mô hình học nhiễu, cột chứa giá trị tương lai làm kết quả đẹp giả, cột không có lúc vận hành làm mô hình vô dụng. Bốn bước giúp chọn đúng cột và khớp RQ: (1) tài liệu cơ quan cho biết ý nghĩa vật lý; (2) bảng thiếu và bảng tương quan trong EDA loại cột thiếu nhiều hoặc trùng; (3) SQL với LAG/LEAD tách quá khứ khỏi tương lai; (4) SHAP sau mô hình xác nhận cột thực sự đóng góp. Mỗi cột giữ lại phải nối về một RQ: cột mô tả cho RQ1, cột dự báo cho RQ2, cột nhóm hoặc điều kiện cho RQ3.

## Bước 1: hiểu bài toán và nạp dữ liệu

### 1.1. Bài toán và ba câu hỏi

Ghi lại ở đây ba câu hỏi nghiên cứu của đề cương (RQ1 mô tả, RQ2 mô hình, RQ3 câu hỏi thứ ba) và biến mục tiêu; mọi bảng và hình sau này phải nối về một trong ba câu hỏi.

**Làm gì**: tải file gốc từ trang cơ quan vào `data/raw/`, nạp bằng DuckDB (đọc thẳng file từ đĩa, chỉ lấy cột và dòng cần) rồi in số dòng, số cột, khoảng thời gian, số đơn vị.

**Kiểm tra**: `df.shape` khớp tài liệu cơ quan; khoảng thời gian đúng; số đơn vị đúng phạm vi đề cương.

In [4]:
# Step 1b (v3): retry khi 503, cache kết quả, kiểm tra tính đầy đủ trước khi giao tập hợp
import pandas as pd, numpy as np, pathlib, time, json
from dataretrieval import nwis

START, END = '2018-01-01', '2026-09-14'
PARAMS = {'wtemp': '00010', 'dox': '00300', 'turb': '63680', 'flow': '00060'}
STATES = ['CA', 'MD', 'VA', 'PA', 'DE', 'NY', 'WV', 'DC']
MIN_DAYS = 0
CACHE = pathlib.Path('data/raw/_cache'); CACHE.mkdir(parents=True, exist_ok=True)

def with_retry(fn, tries=6, base=3, label=''):
    """Gọi fn(); nếu 503/lỗi mạng thì chờ 3, 6, 12, 24, 48 giây rồi thử lại. Trả (kết quả, thành_công)."""
    for k in range(tries):
        try:
            return fn(), True
        except Exception as e:
            msg = f'{type(e).__name__}: {str(e)[:60]}'
            # Lỗi "không có dữ liệu" thật (404/empty) thì không cần retry
            if '404' in msg or 'NoSitesError' in msg:
                return None, True
            wait = base * 2**k
            print(f'  {label} lần {k+1}/{tries} lỗi ({msg}), chờ {wait}s')
            time.sleep(wait)
    return None, False

# --- 1. Danh sách trạm theo (bang, thông số), có cache để không gọi lại ---
def sites_with_param(state, pcode):
    f = CACHE / f'sites_{state}_{pcode}.json'
    if f.exists():
        return set(json.loads(f.read_text())), True
    res, ok = with_retry(lambda: nwis.what_sites(stateCd=state, parameterCd=pcode,
                                                  hasDataTypeCd='dv', siteType='ST'),
                         label=f'[{state}/{pcode}]')
    if not ok:
        return set(), False                       # thất bại thật: KHÔNG cache, KHÔNG coi là "rỗng"
    df = res[0] if res is not None else None
    s = set() if (df is None or df.empty) else set(df['site_no'].astype(str))
    f.write_text(json.dumps(sorted(s)))           # chỉ cache khi server trả lời hợp lệ
    return s, True

super_sites, summary, failed = {}, [], []
for st in STATES:
    per_param, all_ok = {}, True
    for k, v in PARAMS.items():
        s, ok = sites_with_param(st, v)
        per_param[k] = s
        if not ok:
            all_ok = False; failed.append(f'{st}/{v}')
        time.sleep(0.5)                           # giảm tải cho server
    inter = set.intersection(*per_param.values())
    summary.append({'state': st, **{k: len(s) for k, s in per_param.items()},
                    'intersection': len(inter), 'complete': all_ok})
    if all_ok:
        super_sites.update({s: st for s in inter})

print(pd.DataFrame(summary).to_string(index=False))
if failed:
    print('\nCHƯA LẤY ĐƯỢC:', failed)
    print('=> Chạy lại cell này (các truy vấn đã thành công được cache, chỉ gọi lại phần thiếu).')
assert not failed, 'Danh sách trạm chưa đầy đủ do lỗi server. Chạy lại cell trước khi đi tiếp.'
print('Tổng số siêu trạm:', len(super_sites))

state  wtemp  dox  turb  flow  intersection  complete
   CA    462   34    31  2419            11      True
   MD     41   15    32   264            12      True
   VA    215   44    51   403            25      True
   PA    274   91    48   636            23      True
   DE     25   10     4    52             1      True
   NY    102    6    49   684             4      True
   WV    112   18     8   295             6      True
   DC      7    3     6     7             1      True
Tổng số siêu trạm: 83


In [5]:
# Step 1b (v3, cell 2): tải daily values, tính coverage, lấy metadata, xuất CSV
def fetch_dv_site(site):
    f = CACHE / f'dv_{site}.parquet'
    if f.exists():
        return pd.read_parquet(f), True
    res, ok = with_retry(lambda: nwis.get_dv(sites=[site], parameterCd=list(PARAMS.values()),
                                              start=START, end=END), label=f'[dv {site}]')
    if not ok:
        return None, False
    d = res[0] if res is not None else pd.DataFrame()
    if d is not None and not d.empty:
        d = d.reset_index()
        d['site_no'] = site                        # gán trực tiếp, không phụ thuộc cột index
        d.to_parquet(f, index=False)
        return d, True
    return pd.DataFrame(), True

frames, dv_failed = [], []
for i, s in enumerate(sorted(super_sites), 1):
    d, ok = fetch_dv_site(s)
    if not ok: dv_failed.append(s)
    elif d is not None and not d.empty: frames.append(d)
    if i % 5 == 0: print(f'  đã tải {i}/{len(super_sites)} trạm')
    time.sleep(0.3)
if dv_failed:
    print('CHƯA TẢI ĐƯỢC:', dv_failed, '=> chạy lại cell này')
raw = pd.concat(frames, ignore_index=True)
print('raw:', raw.shape, '| trạm có dữ liệu:', raw['site_no'].nunique())

# Coverage: ngày đủ cả 4 thông số (mỗi thông số có ít nhất 1 cột cảm biến có giá trị)
def has_param(df, code):
    cols = [c for c in df.columns if c.startswith(code) and c.endswith('_Mean')]
    return df[cols].notna().any(axis=1) if cols else pd.Series(False, index=df.index)
ok4 = pd.concat({k: has_param(raw, v) for k, v in PARAMS.items()}, axis=1).all(axis=1)
cov = raw.assign(ok=ok4).groupby('site_no')['ok'].sum().astype(int)
print(cov.describe().round(0))

# Metadata
final_sites = sorted(s for s in super_sites if cov.get(s, 0) >= MIN_DAYS)
info_frames = []
for i in range(0, len(final_sites), 20):
    res, ok = with_retry(lambda: nwis.get_info(sites=final_sites[i:i+20]), label=f'[info {i}]')
    if ok and res is not None: info_frames.append(res[0])
info = pd.concat(info_frames, ignore_index=True)
info['site_no'] = info['site_no'].astype(str)
info = info.drop_duplicates('site_no')

out = info[['site_no', 'station_nm', 'dec_lat_va', 'dec_long_va']].copy()
out['state'] = out['site_no'].map(super_sites)
out['n_days_4param'] = out['site_no'].map(cov).fillna(0).astype(int)
out = (out[['site_no', 'station_nm', 'state', 'dec_lat_va', 'dec_long_va', 'n_days_4param']]
       .sort_values(['state', 'n_days_4param'], ascending=[True, False]).reset_index(drop=True))
assert out['site_no'].is_unique
out.to_csv('data/raw/wq_sites_all.csv', index=False)
print(f'\nĐã lưu data/raw/wq_sites_all.csv: {len(out)} trạm')
print(out.groupby('state').agg(n_sites=('site_no', 'count'), median_days=('n_days_4param', 'median')))

  đã tải 5/83 trạm
  đã tải 10/83 trạm
  đã tải 15/83 trạm
  đã tải 20/83 trạm
  đã tải 25/83 trạm
  đã tải 30/83 trạm
  đã tải 35/83 trạm
  đã tải 40/83 trạm
  đã tải 45/83 trạm
  đã tải 50/83 trạm
  đã tải 55/83 trạm
  đã tải 60/83 trạm
  đã tải 65/83 trạm
  đã tải 70/83 trạm
  đã tải 75/83 trạm
  đã tải 80/83 trạm
raw: (210804, 238) | trạm có dữ liệu: 77
count      77.0
mean      893.0
std      1045.0
min         0.0
25%         0.0
50%       508.0
75%      1722.0
max      3135.0
Name: ok, dtype: float64

Đã lưu data/raw/wq_sites_all.csv: 83 trạm
       n_sites  median_days
state                      
CA          11          0.0
DC           1       2993.0
DE           1       2840.0
MD          12       1129.0
NY           4       2614.0
PA          23       1666.0
VA          25          0.0
WV           6          0.0


In [6]:
# 1. Thiết lập tiêu chuẩn (Ngưỡng đậu)
MIN_VALID_DAYS = 500 

# 2. Tìm các trạm ĐẠT YÊU CẦU từ bảng thống kê (out)
valid_sites = out[out['n_days_4param'] >= MIN_VALID_DAYS]['site_no'].tolist()

# 3. Rút trích dữ liệu thực tế của các trạm đó từ cục data thô (raw)
df = raw[raw['site_no'].isin(valid_sites)].copy()

In [7]:
# --- Bước 1c (Tối ưu All-Sites): Thay mã lỗi cơ quan và Coalesce toàn bộ trạm ---
import pandas as pd
import numpy as np

# 1. Lấy TẤT CẢ các trạm đủ yêu cầu thay vì giới hạn 30
# Thiết lập ngưỡng: Chỉ lấy những trạm có ít nhất 500 ngày đủ 4 thông số để mô hình học không bị nhiễu
MIN_VALID_DAYS = 500 
valid_sites = out[out['n_days_4param'] >= MIN_VALID_DAYS]['site_no'].tolist()

df = raw[raw['site_no'].isin(valid_sites)].copy()

# Chuẩn hóa tên cột định danh để khớp với downstream Pipeline
df = df.rename(columns={'site_no': 'station', 'datetime': 'date'})
if 'date' not in df.columns and 'index' in df.columns:
    df = df.rename(columns={'index': 'date'})
df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)

print(f"Đã nạp toàn bộ {len(valid_sites)} trạm đạt chuẩn. Kích thước tập dữ liệu thô: {df.shape}")

# 2. Xử lý mã thiếu cơ quan TRƯỚC khi tính toán (tránh phương sai ảo)
AGENCY_MISSING_CODES = [-999999, -99999, -9999, -999, 999, 9999, 99999, 999999]
numeric_cols = df.select_dtypes(include='number').columns
n_replaced = 0

for c in numeric_cols:
    mask = df[c].isin(AGENCY_MISSING_CODES)
    n_replaced += mask.sum()
    df.loc[mask, c] = np.nan
print(f'Đã thay {n_replaced} giá trị mã lỗi cơ quan bằng NaN')

# 3. Gộp cột (Coalesce) trực tiếp từ mã thông số của USGS
PARAM_MAP = {'wtemp': '00010', 'dox': '00300', 'turb': '63680', 'flow': '00060'}

def coalesce_param(df_source, param_code):
    result = pd.Series(np.nan, index=df_source.index)
    
    # Ưu tiên 1: Cảm biến Mean
    alt_mean = [c for c in df_source.columns if c.startswith(param_code) and c.endswith('_Mean') and not c.endswith('_cd')]
    for c in alt_mean: result = result.fillna(df_source[c])
        
    # Ưu tiên 2: (Max + Min) / 2
    max_cols = [c for c in df_source.columns if c.startswith(param_code) and c.endswith('_Maximum') and not c.endswith('_cd')]
    min_cols = [c for c in df_source.columns if c.startswith(param_code) and c.endswith('_Minimum') and not c.endswith('_cd')]
    bases = set(c.rsplit('_Maximum', 1)[0] for c in max_cols) & set(c.rsplit('_Minimum', 1)[0] for c in min_cols)
    for b in bases: result = result.fillna((df_source[b+'_Maximum'] + df_source[b+'_Minimum']) / 2)
        
    # Ưu tiên 3: Cảm biến Median
    med_cols = [c for c in df_source.columns if c.startswith(param_code) and c.endswith('_Median') and not c.endswith('_cd')]
    for c in med_cols: result = result.fillna(df_source[c])
        
    return result

clean = df[['station', 'date']].copy()
print("\n--- Phục hồi dữ liệu qua Coalesce ---")
for main_name, code in PARAM_MAP.items():
    clean[main_name] = coalesce_param(df, code)
    n_valid = clean[main_name].notna().sum()
    print(f'{main_name} ({code}): có {n_valid} giá trị hợp lệ ({n_valid/len(df):.1%})')

# 4. Cảnh báo khoảng vật lý sinh thái
print("\n--- Kiểm tra giới hạn vật lý ---")
PHYSICAL_RANGES = {'wtemp': (-2, 40), 'dox': (0, 20), 'turb': (0, 5000), 'flow': (0, 1e6)}
for col, (lo, hi) in PHYSICAL_RANGES.items():
    n_bad = ((clean[col] < lo) | (clean[col] > hi)).sum()
    print(f'{col}: {"OK" if n_bad == 0 else f"CẢNH BÁO: {n_bad} giá trị ngoài khoảng [{lo}, {hi}]"}')

# 5. Chốt khung dữ liệu và xuất danh sách ALL SITES
df = clean 
out[out['site_no'].isin(valid_sites)].to_csv('data/raw/wq_sites_all_valid.csv', index=False)

# --- KIỂM TRA ĐẦU RA MỚI (Không ràng buộc 30 trạm) ---
unique_stations = df['station'].unique()
n_stations = len(unique_stations)

print("\n--- KẾT QUẢ NGHIỆM THU DỮ LIỆU ---")
print(f"PASS: Bảng dữ liệu đã gom thành công toàn bộ {n_stations} trạm đạt chuẩn.")
print(f"QUY MÔ: Tổng số dòng thu được là {len(df):,} dòng.")

if len(df) >= 30000:
    print("TEST CASE: Đạt chuẩn yêu cầu > 30.000 dòng của Notebook_A.")
else:
    print("⚠️ CẢNH BÁO: Dữ liệu ít hơn 30.000 dòng, xem lại tham số MIN_VALID_DAYS.")

Đã nạp toàn bộ 39 trạm đạt chuẩn. Kích thước tập dữ liệu thô: (103645, 238)
Đã thay 504 giá trị mã lỗi cơ quan bằng NaN

--- Phục hồi dữ liệu qua Coalesce ---
wtemp (00010): có 86165 giá trị hợp lệ (83.1%)
dox (00300): có 76047 giá trị hợp lệ (73.4%)
turb (63680): có 80368 giá trị hợp lệ (77.5%)
flow (00060): có 103193 giá trị hợp lệ (99.6%)

--- Kiểm tra giới hạn vật lý ---
wtemp: OK
dox: OK
turb: OK
flow: OK

--- KẾT QUẢ NGHIỆM THU DỮ LIỆU ---
PASS: Bảng dữ liệu đã gom thành công toàn bộ 39 trạm đạt chuẩn.
QUY MÔ: Tổng số dòng thu được là 103,645 dòng.
TEST CASE: Đạt chuẩn yêu cầu > 30.000 dòng của Notebook_A.


In [8]:
import pandas as pd

# Đọc danh sách toàn bộ trạm từ file tổng ở Bước 1b
df_all = pd.read_csv('data/raw/wq_sites_all.csv')

# 1. Lọc TẤT CẢ các trạm đạt chuẩn theo ngưỡng thời gian (>= 500 ngày)
MIN_VALID_DAYS = 500
valid_stations = df_all[df_all['n_days_4param'] >= MIN_VALID_DAYS].copy()

# 2. Thống kê chi tiết số trạm đạt chuẩn theo từng bang
print(f"--- THỐNG KÊ {len(valid_stations)} TRẠM ĐẠT CHUẨN THEO BANG ---")
state_summary = valid_stations['state'].value_counts().reset_index()
state_summary.columns = ['Bang', 'Số trạm đạt chuẩn']
print(state_summary.to_string(index=False))
print(f"Tổng số trạm gom được: {len(valid_stations)}")

# 3. Lưu ra file mới hoàn toàn để tránh lỗi PermissionError do trùng khóa file cũ
output_path = 'data/raw/wq_sites_all_valid.csv'
valid_stations.to_csv(output_path, index=False)
print(f"\n✅ Đã xuất thành công toàn bộ danh sách trạm đạt chuẩn vào '{output_path}'!")

--- THỐNG KÊ 39 TRẠM ĐẠT CHUẨN THEO BANG ---
Bang  Số trạm đạt chuẩn
  PA                 22
  MD                  9
  NY                  3
  WV                  2
  DC                  1
  DE                  1
  VA                  1
Tổng số trạm gom được: 39

✅ Đã xuất thành công toàn bộ danh sách trạm đạt chuẩn vào 'data/raw/wq_sites_all_valid.csv'!


### 1.1b. Hồ sơ tự động

`ydata-profiling` trên mẫu 20 nghìn dòng tạo `report/profile.html`; mở bằng trình duyệt để xem nhanh phân phối, thiếu, tương quan trước khi tự viết EDA ở các ô sau.

In [9]:
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 267.96it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

**Làm gì**: chuẩn tên cột; bỏ trùng theo khoá; bỏ thiếu ở biến mục tiêu; loại giá trị ngoài khoảng vật lý theo tài liệu cơ quan; dựng khoá thời gian đầy đủ và nội suy ngắn; ghép nguồn thứ hai; lưu parquet.

**Kiểm tra**: số dòng sau merge không tăng; tỷ lệ khớp trên 90%; ghi số dòng bỏ ở mỗi bước vào nhật ký làm sạch (ô sau).

In [10]:
# Step 2a: coordinates of ALL VALID USGS sites + global GHCN station inventory (Robust version)
import pandas as pd, numpy as np, requests
from dataretrieval import nwis

# 1. Đọc file CSV và tự động bù số '0' ở đầu cho chuẩn USGS
df_sites = pd.read_csv('data/raw/wq_sites_all_valid.csv', dtype={'site_no': str})
SITES = [s.zfill(8) if len(s) == 7 else s for s in df_sites['site_no'].tolist()]

bad = [s for s in SITES if len(s) not in (8, 9, 10, 11, 12, 13, 15)]
assert not bad, f'site_no bat thuong (co the mat so 0 dau): {bad}'

# 2. Tải metadata tọa độ, tự động lọc bỏ các trạm lỗi thay vì bắt chết bằng assert
print("Đang tải thông tin tọa độ từ USGS...")
info_res = nwis.get_info(sites=SITES)
if info_res is not None and not info_res[0].empty:
    meta = info_res[0][['site_no', 'dec_lat_va', 'dec_long_va']].drop_duplicates('site_no').copy()
    meta['site_no'] = meta['site_no'].astype(str)
    
    # Tìm các trạm bị thiếu metadata để thông báo rõ ràng cho bạn biết
    missing_sites = set(SITES) - set(meta['site_no'])
    if missing_sites:
        print(f"⚠️ Cảnh báo: Có {len(missing_sites)} trạm không phản hồi tọa độ từ USGS và đã được tự động loại bỏ: {list(missing_sites)}")
    
    # Cập nhật lại danh sách SITES chính thức chỉ gồm những trạm lấy được tọa độ thành công
    SITES = meta['site_no'].tolist()
else:
    raise RuntimeError("Không thể tải metadata từ USGS. Kiểm tra lại kết nối mạng.")

print(f"✅ Đã xác thực thành công tọa độ cho {len(SITES)} trạm.")

# 3. Tải cache trạm thời tiết GHCN toàn cầu như bình thường
cols = [(0,11),(12,20),(21,30),(31,37),(38,40),(41,71)]
names = ['id','lat','lon','elev','state','name']
stations = pd.read_fwf('https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt', colspecs=cols, names=names)
stations.to_csv('data/raw/ghcnd_stations_cache.csv', index=False)

print(f"Kích thước metadata trạm nước: {meta.shape} | Kích thước trạm thời tiết: {stations.shape}")
print(meta.head())

Đang tải thông tin tọa độ từ USGS...
⚠️ Cảnh báo: Có 2 trạm không phản hồi tọa độ từ USGS và đã được tự động loại bỏ: ['15319015', '205551614']
✅ Đã xác thực thành công tọa độ cho 37 trạm.
Kích thước metadata trạm nước: (37, 3) | Kích thước trạm thời tiết: (133039, 6)
    site_no  dec_lat_va  dec_long_va
0  01438500   41.309167   -74.795278
1  01473500   40.115556   -75.356111
2  01473900   40.123999   -75.219899
3  01474000   40.015290   -75.206834
4  01474500   39.967791   -75.188501


In [11]:
# Step 2b only match against GHCN stations reporting PRCP+TMAX through the current year
from datetime import date
END_YEAR = date.today().year

inv_cols = [(0,11),(12,20),(21,30),(31,35),(36,40),(41,45)]
inv_names = ['id','lat','lon','element','firstyear','lastyear']
inv = pd.read_fwf('https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt', colspecs=inv_cols, names=inv_names)

ok_ids = set(
    inv[(inv.element.isin(['PRCP', 'TMAX'])) & (inv.firstyear <= 2018) & (inv.lastyear >= END_YEAR)]
    .groupby('id').filter(lambda g: g.element.nunique() == 2)
    .id.unique()
)
cand = stations[stations.id.isin(ok_ids)].reset_index(drop=True)
print('nam hien tai dung de loc:', END_YEAR)
print('so tram GHCN du PRCP+TMAX 2018-hien tai:', len(cand))
assert len(cand) > 0, 'khong tim duoc tram GHCN nao du dieu kien'

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1); dl = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

matches = []
for _, s in meta.iterrows():
    d = haversine(s.dec_lat_va, s.dec_long_va, cand.lat.values, cand.lon.values)
    j = np.argmin(d)
    matches.append([s.site_no, cand.iloc[j]['id'], cand.iloc[j]['name'], round(d[j], 1)])
match_df = pd.DataFrame(matches, columns=['station', 'ghcn_id', 'ghcn_name', 'dist_km'])
match_df.to_csv('report/table_ghcn_match.csv', index=False)
print(match_df)
print('\nmax dist_km:', match_df.dist_km.max())

nam hien tai dung de loc: 2026
so tram GHCN du PRCP+TMAX 2018-hien tai: 8766
     station      ghcn_id                      ghcn_name  dist_km
0   01438500  USC00306774                    PORT JERVIS     12.5
1   01473500  USC00366370                     NORRISTOWN      1.8
2   01473900  USC00366370                     NORRISTOWN     10.1
3   01474000  USC00366886            PHILA FRANKLIN INST      6.9
4   01474500  USC00366886            PHILA FRANKLIN INST      1.7
5   01480617  USW00003761                   AVONDALE 2 N     11.5
6   01480870  USW00003761                   AVONDALE 2 N     15.5
7   01481000  USC00079605         WILMINGTON PORTER RSCH     11.6
8   01481500  USC00079605         WILMINGTON PORTER RSCH      3.1
9   01491000  USC00072730                          DOVER     29.4
10  01493112  USW00013701          ABERDEEN PHILLIPS FLD     31.0
11  01573208  USC00364896                    LEBANON 2 W     14.3
12  01573660  USW00014711  MIDDLETOWN HARRISBURG INTL AP      5.0

In [12]:
# Step 2c: download and parse .dly for the unique GHCN stations in match_df, map back to USGS station
import time, requests
from datetime import date

# ĐÃ SỬA: Cố định ngày kết thúc là 2026-09-14 theo đúng cấu hình ở Bước 1b
END_DATE = '2026-09-14'   

unique_ghcn = match_df.ghcn_id.unique().tolist()
print(f'So tram GHCN duy nhat can tai: {len(unique_ghcn)} (tu {len(match_df)} tram USGS)')

def parse_dly(ghcn_id):
    url = f'https://www.ncei.noaa.gov/pub/data/ghcn/daily/all/{ghcn_id}.dly'
    r = requests.get(url, timeout=30, headers={'User-Agent': 'ady201m-coursework'})
    r.raise_for_status()
    rows = []
    for ln in r.text.splitlines():
        el = ln[17:21]
        if el not in ('PRCP', 'TMAX'):
            continue
        yr, mo = int(ln[11:15]), int(ln[15:17])
        for day in range(31):
            val = int(ln[21 + day*8 : 26 + day*8])
            if val == -9999:
                continue
            try:
                d = pd.Timestamp(yr, mo, day + 1)
            except ValueError:
                continue
            rows.append([d, el, val])
    return pd.DataFrame(rows, columns=['date', 'element', 'value'])

# --- tai tung tram GHCN duy nhat, khong dung ca vong lap neu 1 tram loi ---
weather = {}
failed = []
for gid in unique_ghcn:
    try:
        d = parse_dly(gid)
        d = d[(d.date >= '2018-01-01') & (d.date <= END_DATE)]
        wide = d.pivot_table(index='date', columns='element', values='value', aggfunc='first').reset_index()
        wide['PRCP'] = wide.get('PRCP', np.nan) / 10.0   # tenths of mm -> mm
        wide['TMAX'] = wide.get('TMAX', np.nan) / 10.0   # tenths of degC -> degC
        weather[gid] = wide[['date', 'PRCP', 'TMAX']]
        print(f'{gid}: OK, {len(wide)} ngay')
    except Exception as e:
        failed.append((gid, str(e)))
        print(f'{gid}: LOI - {e}')
    time.sleep(0.3)   # nhe tay voi NOAA, tranh bi chan tam thoi

print(f'\nTai xong: {len(weather)}/{len(unique_ghcn)} tram GHCN')
if failed:
    print('Cac tram loi (can tai lai rieng):', failed)
# --- anh xa nguoc: ghcn_id -> nhieu station USGS, theo match_df ---
frames = []
for _, r in match_df.iterrows():
    if r.ghcn_id not in weather:
        print(f'BO QUA {r.station}: chua tai duoc weather cho {r.ghcn_id}')
        continue
    w = weather[r.ghcn_id].copy()
    w['station'] = r.station
    frames.append(w[['station', 'date', 'PRCP', 'TMAX']])

ghcn = pd.concat(frames, ignore_index=True)
ghcn.to_csv('data/raw/ghcn_wq_daily.csv', index=False)
print(ghcn.shape, 'so tram trong file:', ghcn.station.nunique())

So tram GHCN duy nhat can tai: 25 (tu 37 tram USGS)
USC00306774: OK, 3174 ngay
USC00366370: OK, 3123 ngay
USC00366886: OK, 2973 ngay
USW00003761: OK, 3173 ngay
USC00079605: OK, 3074 ngay
USC00072730: OK, 2921 ngay
USW00013701: OK, 3175 ngay
USC00364896: OK, 3133 ngay
USW00014711: OK, 3179 ngay
USC00367732: OK, 3179 ngay
USC00182060: OK, 3160 ngay
USC00467730: OK, 2512 ngay
USC00444909: OK, 3166 ngay
USC00182906: OK, 3179 ngay
USC00182336: OK, 3177 ngay
USC00182325: OK, 2628 ngay
USC00180700: OK, 3139 ngay
USC00186350: OK, 2990 ngay
USC00361526: OK, 3133 ngay
USW00003872: OK, 3179 ngay
USC00368361: OK, 3179 ngay
USW00014860: OK, 3179 ngay
USC00306525: OK, 3175 ngay
USW00014768: OK, 3179 ngay
USC00306314: OK, 3179 ngay

Tai xong: 25/25 tram GHCN
(114350, 4) so tram trong file: 37


In [13]:
# Testcases cho step 2c (Đã sửa để không hardcode số trạm)
assert ghcn.station.nunique() == len(match_df), f'thieu tram: chi co {ghcn.station.nunique()}/{len(match_df)}'
assert not ghcn.duplicated(['station', 'date']).any(), 'trung khoa station-date'
miss_prcp = ghcn.PRCP.isna().mean(); miss_tmax = ghcn.TMAX.isna().mean()
print(f'ty le thieu PRCP: {miss_prcp:.1%} | TMAX: {miss_tmax:.1%}')
assert miss_prcp < 0.3, f'PRCP thieu qua nhieu: {miss_prcp:.1%}'
assert miss_tmax < 0.3, f'TMAX thieu qua nhieu: {miss_tmax:.1%}'
print('Tests 2c: OK')

ty le thieu PRCP: 3.7% | TMAX: 1.8%
Tests 2c: OK


In [14]:
# Step 2 (merge chính thức): Cố định chuẩn xác 37 trạm đã được xác thực tọa độ từ Bước 2a
keep = [c for c in ['turb','dox','wtemp','flow'] if c in df.columns]

# Đảm bảo chỉ lấy đúng các trạm có trong danh sách SITES chuẩn (37 trạm)
df = df[df['station'].isin(SITES)].copy()

df_clean = df.drop_duplicates(['station', 'date']).dropna(subset=['wtemp'])
excluded = sorted(set(df['station'].unique()) - set(df_clean['station'].unique()))
N_STATIONS = df_clean['station'].nunique()

print(f'Số trạm chuẩn chính thức đưa vào xử lý: {N_STATIONS}')
assert N_STATIONS == len(SITES), f'Lỗi lệch số trạm: {N_STATIONS} != {len(SITES)}'

if excluded:
    print(f'Tự động loại {len(excluded)} trạm không có wtemp trong giai đoạn: {excluded}')

full = df_clean.set_index('date').groupby('station')[keep].resample('D').mean().reset_index()
full[keep] = full.groupby('station')[keep].transform(lambda s: s.interpolate(limit=3))

w = pd.read_csv('data/raw/ghcn_wq_daily.csv', parse_dates=['date'], dtype={'station': str})
m = full.merge(w, on=['station', 'date'], how='left')
m['do_low'] = (m.dox < 5).astype(int) if 'dox' in keep else 0
m.to_parquet('data/processed/wq.parquet', index=False)

match_rate = m['PRCP'].notna().mean()
assert m.station.nunique() == N_STATIONS, 'mất thêm trạm ngoài dự kiến trong bước merge'
assert match_rate > 0.85, f'tỷ lệ khớp GHCN dưới ngưỡng: {match_rate:.1%}'

print('m.shape:', m.shape, '| tỷ lệ khớp GHCN:', round(match_rate, 3))
print(f'Bước 2: OK, chính thức chốt chuẩn {N_STATIONS} trạm từ đây.')

Số trạm chuẩn chính thức đưa vào xử lý: 37
m.shape: (86444, 9) | tỷ lệ khớp GHCN: 0.932
Bước 2: OK, chính thức chốt chuẩn 37 trạm từ đây.


### 2.1. Nhật ký làm sạch (bảng đưa vào bài)

Mỗi bước: tên bước, số dòng trước, số dòng sau, số bỏ, lý do có tài liệu.

In [15]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])
_, cleaning_log = clean_log(m, [
    ('drop duplicates', lambda x: x.drop_duplicates(['station', 'date']), 'same unit and timestamp'),
    ('drop missing target', lambda x: x.dropna(subset=['wtemp']), 'cannot be forecast'),
])
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)

                  step  rows_before  rows_after  dropped                   reason
0      drop duplicates        86444       86444        0  same unit and timestamp
1  drop missing target        86444       83722     2722       cannot be forecast


### 1.2. EDA phần 1: cấu trúc bảng (info, describe) và cách đọc

**Làm gì**: `info()` cho biết kiểu dữ liệu và số ô không thiếu của từng cột; `describe()` cho thống kê năm số và trung bình; với cột phân loại xem số giá trị khác nhau và giá trị phổ biến nhất.

**Đọc thế nào**: cột số mà `dtype` là object nghĩa là có ký tự lạ cần ép kiểu; `max` bất thường (ví dụ 999, -9999) là mã thiếu của cơ quan; `std` bằng 0 là cột hằng, bỏ. Ghi lại các phát hiện này vào bảng `report/table_eda_notes.csv` để đưa vào mục Data của bài.

In [16]:
eda = m.copy()
import io
buf = io.StringIO(); eda.info(buf=buf); print(buf.getvalue()[:3000])
desc_num = eda.describe().T                         # numeric statistics
desc_num['skew'] = eda.select_dtypes('number').skew()  # skewness, > 1 means strongly right-skewed
desc_num.round(3).to_csv('report/table_describe_numeric.csv'); print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(exclude='number').describe().T   # categorical columns: count, unique, top, freq
desc_cat.to_csv('report/table_describe_categorical.csv'); print(desc_cat.head(15))
notes = []                                          # record EDA notes for the paper
for c in eda.select_dtypes('number').columns:
    s = eda[c]
    if s.std() == 0: notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999): notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2: notes.append([c, 'strongly skewed, consider log or tree models'])
pd.DataFrame(notes, columns=['column', 'note']).to_csv('report/table_eda_notes.csv', index=False); print(notes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86444 entries, 0 to 86443
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   station  86444 non-null  object        
 1   date     86444 non-null  datetime64[ns]
 2   turb     79791 non-null  float64       
 3   dox      74032 non-null  float64       
 4   wtemp    83722 non-null  float64       
 5   flow     83698 non-null  float64       
 6   PRCP     80593 non-null  float64       
 7   TMAX     81683 non-null  float64       
 8   do_low   86444 non-null  int64         
dtypes: datetime64[ns](1), float64(6), int64(1), object(1)
memory usage: 5.9+ MB

          count                           mean                  min                  25%                  50%                  75%                  max  \
date      86444  2022-12-02 03:44:33.138679296  2018-01-01 00:00:00  2020-12-17 00:00:00  2023-03-05 00:00:00  2025-01-03 00:00:00  2026-09-14 00:00:00   
turb 

### 1.3. EDA phần 2: thiếu dữ liệu, mẫu thiếu và quyết định xử lý

**Cách xem**: tỷ lệ thiếu từng cột; thiếu theo thời gian (có tháng nào mất hẳn không); thiếu theo đơn vị (trạm, pin, bang); thiếu có đi cùng nhau không (ma trận đồng thiếu).

**Quyết định xử lý** (ghi vào bảng và vào bài):
- Thiếu ở **biến mục tiêu**: bỏ dòng, không bao giờ điền.
- Thiếu **ngắn** trong chuỗi thời gian (1-3 bước): nội suy tuyến tính theo từng đơn vị.
- Thiếu **dài** hoặc cả tháng: bỏ đoạn đó của đơn vị, ghi lý do.
- Cột thiếu trên 40%: bỏ cột hoặc chỉ giữ cờ có/không.
- Cột phân loại thiếu: gộp vào lớp `unknown`.
- Với mô hình cây (LightGBM, XGBoost) thiếu trong đặc trưng có thể để nguyên, mô hình tự xử lý; với ridge phải điền (median) sau khi chia train/test.

In [17]:
miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select([miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
                             ['keep', 'interpolate/impute median', 'drop or keep flag'], 'drop column')
miss.round(4).to_csv('report/table_missing.csv'); print(miss.head(15))
# missingness of the target by time and by unit
tcol, gcol = 'date', 'station'
by_time = eda.groupby(pd.to_datetime(eda[tcol], errors='coerce').dt.to_period('M') if not np.issubdtype(eda[tcol].dtype, np.number) else eda[tcol])['wtemp'].apply(lambda s: s.isna().mean())
by_unit = eda.groupby(gcol)['wtemp'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
print('target missing by period (top):', by_time.sort_values(ascending=False).head(5).round(3).to_dict())
print('target missing by unit (top):', by_unit.head(5).round(3).to_dict())
# co-missingness matrix: do columns go missing together
co = eda.isna().astype(int); co = co.loc[:, co.sum() > 0]
if co.shape[1] > 1: print(co.corr().round(2))
fig, ax = plt.subplots(figsize=(7, 3))
miss.missing_rate.head(12).plot.bar(ax=ax, color='#1B6B6D'); ax.set_ylabel('Missing rate'); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig('report/fig_eda_missing.png', dpi=300)

         missing_rate                   decision
dox          0.143584          drop or keep flag
turb         0.076963          drop or keep flag
PRCP         0.067685          drop or keep flag
TMAX         0.055076          drop or keep flag
flow         0.031766  interpolate/impute median
wtemp        0.031489  interpolate/impute median
station      0.000000                       keep
date         0.000000                       keep
do_low       0.000000                       keep
target missing by period (top): {Period('2026-02', 'M'): 0.201, Period('2019-01', 'M'): 0.186, Period('2025-01', 'M'): 0.147, Period('2026-01', 'M'): 0.127, Period('2024-01', 'M'): 0.122}
target missing by unit (top): {'04213152': 0.383, '01474000': 0.268, '01473900': 0.267, '01473500': 0.196, '01480870': 0.028}
       turb   dox  wtemp  flow  PRCP  TMAX
turb   1.00  0.52   0.62  0.63  0.01  0.01
dox    0.52  1.00   0.44  0.44 -0.04 -0.01
wtemp  0.62  0.44   1.00  1.00  0.01 -0.02
flow   0.63  0.44   1.00

### 1.4. EDA phần 3: chọn cột và thống kê theo nhóm, theo thời gian

**Chọn cột** theo ba tiêu chí: (1) có ý nghĩa vật lý hoặc nghiệp vụ với mục tiêu; (2) có sẵn tại thời điểm dự báo (không phải giá trị tương lai); (3) không trùng lặp với cột khác (tương quan trên 0,95 thì giữ một). Bảng `table_columns.csv` ghi cột nào giữ, cột nào bỏ và vì sao; bảng này là một phần của mục Data trong bài.

**Thống kê**: trung bình, trung vị, phân vị 5 và 95 của mục tiêu theo nhóm và theo thời kỳ; đây là bảng đầu tiên trả lời RQ1 trước khi làm SQL.

In [18]:
num_cols = [c for c in ['wtemp', 'dox', 'turb', 'flow'] if c in eda.columns]
corr = eda[num_cols].corr()
corr.round(3).to_csv('report/table_corr.csv'); print(corr.round(3))
cols = []
for c in eda.columns:
    if c == 'wtemp': cols.append([c, 'target', 'keep']); continue
    if c in ('date', 'station'): cols.append([c, 'time/unit key', 'keep for feature building']); continue
    if c in num_cols:
        r = corr.loc[c, 'wtemp'] if 'wtemp' in corr.columns else np.nan
        cols.append([c, f'correlation with target {r:.2f}', 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'])
    else: cols.append([c, 'other', 'review'])
pd.DataFrame(cols, columns=['column', 'reason', 'decision']).to_csv('report/table_columns.csv', index=False)
# target statistics by unit and by period
q = lambda s: pd.Series({'mean': s.mean(), 'median': s.median(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95), 'n': s.count()})
by_g = eda.groupby('station')['wtemp'].apply(q).unstack().sort_values('mean', ascending=False)
by_g.round(3).to_csv('report/table_target_by_unit.csv'); print(by_g.head(10).round(3))
period = eda['date'] if np.issubdtype(eda['date'].dtype, np.number) else pd.to_datetime(eda['date'], errors='coerce').dt.month
by_t = eda.groupby(period)['wtemp'].apply(q).unstack()
by_t.round(3).to_csv('report/table_target_by_period.csv'); print(by_t.round(3))

       wtemp    dox   turb   flow
wtemp  1.000 -0.933 -0.037 -0.079
dox   -0.933  1.000  0.016  0.140
turb  -0.037  0.016  1.000  0.195
flow  -0.079  0.140  0.195  1.000
            mean  median    p05    p95       n
station                                       
01473500  17.814    18.7  5.430  27.90  2507.0
04213152  17.522    18.6  8.191  24.50  1886.0
01473900  17.378    18.4  8.000  25.19   792.0
01474000  17.218    18.2  7.650  25.20   791.0
01646500  16.121    16.4  1.935  29.60  3168.0
03185400  16.039    16.1  3.350  28.10  2623.0
01643580  16.024    16.9  1.200  28.50  1173.0
01638500  16.018    16.5  1.100  28.90  1956.0
01491000  15.863    17.4  1.710  26.80  1243.0
01649500  15.301    15.6  2.700  26.70  3168.0
        mean  median   p05     p95       n
date                                      
1      2.909     2.5   0.0   7.400  6572.0
2      3.840     3.8   0.0   8.300  6017.0
3      8.073     8.2   2.7  12.822  7212.0
4     12.842    13.0   6.5  18.600  7315.0
5     17

## Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

**Làm gì**: đưa parquet vào DuckDB; Q1 và Q2 tổng hợp trả lời RQ1 (GROUP BY, window, CTE); Q3 tạo đặc trưng trễ, trung bình trượt và mục tiêu tương lai bằng LAG, LEAD, AVG OVER; Q4 kiểm tra chất lượng. Lưu toàn bộ vào `sql/queries.sql`; mỗi truy vấn SELECT được lưu thành `report/table_q*.csv`.

**Kiểm tra**: chọn một đơn vị và 10 dòng, tính tay trung bình trượt rồi so với SQL; không đặc trưng nào dùng giá trị sau thời điểm dự báo.

In [31]:
# Buoc 3: SQL tra loi RQ1 (Q1-Q4 day du) + kiem dinh thong ke - GON 1 CELL
import duckdb, pandas as pd, numpy as np
from scipy.stats import kruskal, spearmanr

# --- 3.0: nap du lieu, loc khoang vat ly bang SQL (thay NaN, khong xoa dong) ---
con = duckdb.connect('data/processed/ady.duckdb')
con.execute("CREATE OR REPLACE TABLE wq_raw AS SELECT * FROM 'data/processed/wq.parquet'")
con.execute("""
CREATE OR REPLACE TABLE wq AS
SELECT station, date,
       CASE WHEN wtemp BETWEEN -2 AND 40      THEN wtemp ELSE NULL END AS wtemp,
       CASE WHEN dox   BETWEEN 0  AND 20      THEN dox   ELSE NULL END AS dox,
       CASE WHEN turb  BETWEEN 0  AND 5000    THEN turb  ELSE NULL END AS turb,
       CASE WHEN flow  BETWEEN 0  AND 1000000 THEN flow  ELSE NULL END AS flow,
       PRCP, TMAX, do_low
FROM wq_raw
""")

site_meta = pd.read_csv('data/raw/wq_sites_all_valid.csv', dtype={'site_no': str}).rename(columns={'site_no': 'station'})
con.register('site_meta_df', site_meta)
con.execute("CREATE OR REPLACE TABLE site_meta AS SELECT station, station_nm, state FROM site_meta_df")
n_stations = con.execute("SELECT COUNT(DISTINCT station) FROM wq").fetchone()[0]
print('So tram dung trong Buoc 3:', n_stations)

# --- Q1: ngay oxy thap + dinh duc theo tram-nam ---
con.execute("""
CREATE OR REPLACE TABLE q1 AS
SELECT station, year(date) AS yr, SUM(do_low) AS low_do_days, MAX(turb) AS turb_max, AVG(wtemp) AS wtemp_mean
FROM wq GROUP BY station, year(date) HAVING SUM(do_low) > 0 ORDER BY low_do_days DESC, station, yr
""")
q1 = con.execute("SELECT * FROM q1").df(); q1.to_csv('report/table_q1.csv', index=False)
print('\n--- Q1 ---'); print(q1.head(10))

# --- Q2: duc sau mua, JOIN voi site_meta, so sanh theo bang ---
con.execute("""
CREATE OR REPLACE TABLE q2 AS
WITH rain_next AS (
    SELECT station, date, PRCP, LEAD(turb, 1) OVER (PARTITION BY station ORDER BY date) AS turb_next
    FROM wq
)
SELECT sm.state, (r.PRCP > 10) AS rain, ROUND(AVG(r.turb_next), 2) AS turb_next_day, COUNT(r.turb_next) AS n_days
FROM rain_next r JOIN site_meta sm ON r.station = lpad(sm.station, 8, '0')
WHERE r.PRCP IS NOT NULL
GROUP BY sm.state, (r.PRCP > 10) ORDER BY sm.state, rain
""")
q2 = con.execute("SELECT * FROM q2").df(); q2.to_csv('report/table_q2.csv', index=False)
print('\n--- Q2 ---'); print(q2)

# --- Q3: dac trung tre + muc tieu tuong lai (LAG, LEAD, AVG OVER) ---
con.execute("""
CREATE OR REPLACE TABLE feat AS
SELECT station, date, wtemp, dox, turb, flow, PRCP, TMAX, do_low, dayofyear(date) AS doy, month(date) AS mon,
       LAG(wtemp, 1) OVER w AS wt_lag1, LAG(wtemp, 7) OVER w AS wt_lag7,
       AVG(wtemp) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS wt_ma7,
       LAG(dox, 1) OVER w AS do_lag1, AVG(dox) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS do_ma7,
       AVG(TMAX) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS tmax3,
       SUM(PRCP) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS prcp3,
       LEAD(wtemp, 1) OVER w AS y_wt_1d, LEAD(wtemp, 3) OVER w AS y_wt_3d, LEAD(dox, 1) OVER w AS y_do_1d,
       MAX(do_low) OVER (w ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING) AS y_low_3d
FROM wq WINDOW w AS (PARTITION BY station ORDER BY date)
""")
q3_check = con.execute("SELECT COUNT(*) AS n, COUNT(y_wt_3d) AS n3 FROM feat").df()
print('\n--- Q3 check ---'); print(q3_check)
sample_check = con.execute("SELECT * FROM feat WHERE station = (SELECT station FROM feat LIMIT 1) ORDER BY date LIMIT 10").df()
print('Kiem tra tay 10 dong 1 tram:'); print(sample_check[['date','wtemp','wt_lag1','wt_lag7','wt_ma7']])

# --- Q4: kiem tra chat luong (thieu, trung, ngoai khoang) ---
con.execute("""
CREATE OR REPLACE TABLE q4 AS
SELECT
  (SELECT COUNT(*) FROM wq_raw) AS n_rows_wq,
  (SELECT COUNT(*) FROM wq_raw) - (SELECT COUNT(*) FROM (SELECT DISTINCT station, date FROM wq_raw)) AS n_duplicate_keys,
  (SELECT COUNT(*) FROM wq_raw WHERE wtemp IS NOT NULL AND wtemp NOT BETWEEN -2 AND 40) AS wtemp_out_of_range,
  (SELECT COUNT(*) FROM wq_raw WHERE dox   IS NOT NULL AND dox   NOT BETWEEN 0  AND 20) AS dox_out_of_range,
  (SELECT COUNT(*) FROM wq_raw WHERE turb  IS NOT NULL AND turb  NOT BETWEEN 0  AND 5000) AS turb_out_of_range,
  (SELECT COUNT(*) FROM wq_raw WHERE flow  IS NOT NULL AND flow  NOT BETWEEN 0  AND 1000000) AS flow_out_of_range,
  (SELECT SUM(CASE WHEN wtemp IS NULL THEN 1 ELSE 0 END) FROM wq) AS wtemp_missing_after_filter,
  (SELECT SUM(CASE WHEN dox   IS NULL THEN 1 ELSE 0 END) FROM wq) AS dox_missing_after_filter,
  (SELECT SUM(CASE WHEN PRCP  IS NULL THEN 1 ELSE 0 END) FROM wq) AS prcp_missing
""")
q4 = con.execute("SELECT * FROM q4").df(); q4.to_csv('report/table_q4.csv', index=False)
print('\n--- Q4 ---'); print(q4.T)
assert q4.n_duplicate_keys[0] == 0, 'con trung khoa station-date, kiem tra lai Buoc 2'

# --- 3.2: tra loi RQ1 bang pandas + kiem dinh thong ke ---
q1_df = con.execute("SELECT * FROM feat").df()
q1_df['period'] = pd.to_datetime(q1_df['date']).dt.month

rq1a = q1_df.groupby('period')['wtemp'].agg(['mean', 'median', 'std', 'count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv')
print('\n--- RQ1a: theo thang ---'); print(rq1a)

groups = [s.values for _, s in q1_df.groupby('period')['wtemp'] if len(s) > 30]
kw_stat, kw_p = kruskal(*groups)
print(f'\nKruskal-Wallis giua cac thang: H={kw_stat:.2f}, p={kw_p:.4g}')

ext = [c for c in q1_df.select_dtypes('number').columns if c not in ('wtemp',) and not c.startswith('y_wt')]
rows = []
for c in ext:
    r, pv = spearmanr(q1_df[c], q1_df['wtemp'], nan_policy='omit')
    rows.append([c, round(r, 3), pv])
rq1b = pd.DataFrame(rows, columns=['variable', 'spearman_r', 'p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False)
print('\n--- RQ1b: tuong quan Spearman ---'); print(rq1b)

rq1c = q1_df.groupby('station')['wtemp'].agg(['mean', 'count']).sort_values('mean', ascending=False)
pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3).to_csv('report/table_rq1c_units.csv')
print('\n--- RQ1c: top/bottom 5 tram ---'); print(pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3))

hi, lo = rq1a['mean'].idxmax(), rq1a['mean'].idxmin()
ratio = rq1a['mean'].max() / max(rq1a['mean'].min(), 1e-9)
print(f'\nKET LUAN RQ1: thang {hi} co wtemp trung binh cao gap {ratio:.2f} lan thang {lo}, Kruskal-Wallis p={kw_p:.4g}.')
print('\nBuoc 3: xong. Da luu table_q1..q4, table_rq1a/b/c vao report/.')

So tram dung trong Buoc 3: 37

--- Q1 ---
    station    yr  low_do_days  turb_max  wtemp_mean
0  01649500  2019         14.0     139.0   15.359328
1  01491000  2026          6.0      25.9   16.614591
2  01473900  2024          3.0      58.0   17.282741
3  01473900  2026          1.0      51.1   19.474324
4  01491000  2025          1.0      34.0   14.467671

--- Q2 ---
   state   rain  turb_next_day  n_days
0     DC  False          12.91    2157
1     DC   True          23.66     286
2     DE  False           4.61    2371
3     DE   True          27.59     287
4     MD  False           8.67   14460
5     MD   True          19.37    1595
6     NY  False          38.60    7760
7     NY   True          58.79     868
8     PA  False           6.55   35679
9     PA   True          21.66    4005
10    WV  False           7.87    4443
11    WV   True          29.23     500

--- Q3 check ---
       n     n3
0  86444  83611
Kiem tra tay 10 dong 1 tram:
        date  wtemp  wt_lag1  wt_lag7     

In [32]:
# Q3: window AVG OVER + LAG (1 trạm, 10 ngày)
q3 = con.execute("""
WITH w AS (
    SELECT station, date, wtemp,
           AVG(wtemp) OVER (PARTITION BY station ORDER BY date
                            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS wt_ma7,
           wtemp - LAG(wtemp, 1) OVER (PARTITION BY station ORDER BY date) AS d1
    FROM wq
    WHERE station = '01473500'
)
SELECT date, wtemp, ROUND(wt_ma7, 2) AS wt_ma7, ROUND(d1, 2) AS d1
FROM w
WHERE date BETWEEN '2023-07-01' AND '2023-07-10'
ORDER BY date
""").df()
q3.to_csv('report/table_q3.csv', index=False)
print('--- Q3 ---', len(q3), 'dòng'); print(q3)

# Q5: RANK, 3 trạm ấm nhất mỗi bang
q5 = con.execute("""
SELECT sm.state, w.station, ROUND(AVG(w.wtemp), 2) AS wt_mean,
       RANK() OVER (PARTITION BY sm.state ORDER BY AVG(w.wtemp) DESC) AS rk
FROM wq w JOIN site_meta sm ON w.station = lpad(sm.station, 8, '0')
GROUP BY sm.state, w.station
QUALIFY rk <= 3
ORDER BY sm.state, rk
""").df()
q5.to_csv('report/table_q5.csv', index=False)
print('--- Q5 ---', len(q5), 'dòng'); print(q5)

# Q6: trạm-tháng nóng bất thường so với các trạm khác cùng tháng
q6 = con.execute("""
WITH m AS (
    SELECT station, month(date) AS mon, AVG(wtemp) AS mu
    FROM wq GROUP BY station, month(date)
),
z AS (
    SELECT station, mon, mu,
           AVG(mu) OVER (PARTITION BY mon) AS mon_mean,
           STDDEV(mu) OVER (PARTITION BY mon) AS mon_sd
    FROM m
)
SELECT station, mon, ROUND(mu, 2) AS mu, ROUND((mu - mon_mean) / mon_sd, 2) AS z_score
FROM z
WHERE (mu - mon_mean) / mon_sd > 1
ORDER BY z_score DESC
LIMIT 10
""").df()
q6.to_csv('report/table_q6.csv', index=False)
print('--- Q6 ---', len(q6), 'dòng'); print(q6)

--- Q3 --- 10 dòng
        date  wtemp  wt_ma7   d1
0 2023-07-01   25.0   23.47  1.0
1 2023-07-02   25.5   23.89  0.5
2 2023-07-03   26.4   24.31  0.9
3 2023-07-04   27.4   24.97  1.0
4 2023-07-05   28.0   25.66  0.6
5 2023-07-06   29.1   26.49  1.1
6 2023-07-07   29.3   27.24  0.2
7 2023-07-08   28.3   27.71 -1.0
8 2023-07-09   26.1   27.80 -2.2
9 2023-07-10   22.7   27.27 -3.4
--- Q5 --- 13 dòng
   state   station  wt_mean  rk
0     DC  01648010    14.66   1
1     DE  01481500    14.20   1
2     MD  01646500    16.12   1
3     MD  01643580    16.02   2
4     MD  01638500    16.02   3
5     NY  04249000    12.73   1
6     NY  04231600    12.23   2
7     NY  04213500    11.66   3
8     PA  01473500    17.81   1
9     PA  04213152    17.52   2
10    PA  01473900    17.38   3
11    WV  03185400    16.04   1
12    WV  01608500    14.87   2
--- Q6 --- 10 dòng
    station  mon     mu  z_score
0  01493112    1   6.58     2.82
1  01493112   12   8.50     2.73
2  01493112    2   8.07     2.48


In [33]:
q6 = con.execute("""
WITH m AS (
    SELECT station, month(date) AS mon, AVG(wtemp) AS mu
    FROM wq
    GROUP BY station, month(date)
)
SELECT station, mon, ROUND(mu, 2) AS mu
FROM m
WHERE mu > (SELECT AVG(wtemp) + STDDEV(wtemp)
            FROM wq
            WHERE month(date) = m.mon)
ORDER BY mu DESC
LIMIT 10
""").df()

q6.to_csv('report/table_q6.csv', index=False)
print('--- Q6 ---', len(q6), 'dòng')
print(q6)

--- Q6 --- 10 dòng
    station  mon     mu
0  01646500    7  29.19
1  01638500    7  28.42
2  01643580    7  27.94
3  01646500    8  27.63
4  01638500    8  26.98
5  01646500    6  25.24
6  01646500    9  24.02
7  03185400    9  23.63
8  01493112   12   8.50
9  01493112    2   8.07


In [29]:
orig = con.execute("""
WITH m AS (
    SELECT station, month(date) AS mon, AVG(wtemp) AS mu
    FROM wq GROUP BY station, month(date)
)
SELECT station, mon, ROUND(mu,2) AS mu
FROM m
WHERE mu > (SELECT AVG(wtemp) + 2*STDDEV(wtemp) FROM wq)
ORDER BY mu DESC
""").df()
print('Q6 gốc trả về', len(orig), 'dòng')

near = con.execute("""
WITH m AS (
    SELECT station, month(date) AS mon, AVG(wtemp) AS mu
    FROM wq GROUP BY station, month(date)
)
SELECT station, mon, ROUND(mu,2) AS mu,
       ROUND((SELECT AVG(wtemp) + 2*STDDEV(wtemp) FROM wq) - mu, 2) AS gap_to_threshold
FROM m ORDER BY mu DESC LIMIT 3
""").df()
print(near)
near.to_csv('report/table_q6_global_check.csv', index=False)

Q6 gốc trả về 0 dòng
    station  mon     mu  gap_to_threshold
0  01646500    7  29.19              1.20
1  01638500    7  28.42              1.98
2  01643580    7  27.94              2.45


### 3.2. Tiếp nối SQL: bảng trả lời RQ1 bằng pandas và kiểm định

**Làm gì**: từ kết quả Q1, Q2 dựng bảng có số thứ tự đưa thẳng vào bài; kiểm định khác biệt giữa nhóm (Kruskal-Wallis, không cần giả định chuẩn) và tương quan Spearman giữa mục tiêu và các biến ngoại sinh kèm p-value.

**Đọc thế nào**: p < 0,05 nghĩa là khác biệt giữa nhóm khó do ngẫu nhiên; hệ số Spearman cho chiều và độ mạnh quan hệ đơn điệu; kết luận RQ1 phải là một câu có số, ví dụ "mùa đông cao gấp 1,9 lần mùa hè, p < 0,001".

In [20]:
from scipy.stats import kruskal, spearmanr
q1 = con.execute("SELECT * FROM feat").df()          # feature table created in Q3
if np.issubdtype(q1['date'].dtype, np.number): q1['period'] = q1['date']            # numeric time key (year, cycle)
else: q1['period'] = pd.to_datetime(q1['date'], errors='coerce').dt.month             # datetime key: use month as period
# Table RQ1a: target by period
rq1a = q1.groupby('period')['wtemp'].agg(['mean', 'median', 'std', 'count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv'); print(rq1a)
groups = [s.dropna().values for _, s in q1_df.groupby('period')['wtemp'] if s.notna().sum() > 30]
kw_stat, kw_p = kruskal(*groups)
print(f'\nKruskal-Wallis giua cac thang: H={kw_stat:.2f}, p={kw_p:.4g}')
# Table RQ1b: Spearman correlation of the target with exogenous variables
ext = [c for c in q1.select_dtypes('number').columns if c not in ('wtemp', 'y_wt_3d') and not c.startswith('wtemp')]
rows = []
for c in ext[:15]:
    r, pv = spearmanr(q1[c], q1['wtemp'], nan_policy='omit'); rows.append([c, round(r, 3), pv])
rq1b = pd.DataFrame(rows, columns=['variable', 'spearman_r', 'p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False); print(rq1b)
# Table RQ1c: target by unit, top and bottom 5
rq1c = q1.groupby('station')['wtemp'].agg(['mean', 'count']).sort_values('mean', ascending=False)
pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3).to_csv('report/table_rq1c_units.csv')
print('RQ1 CONCLUSION (fill with real numbers): highest period', rq1a['mean'].idxmax(), 'is', round(rq1a['mean'].max() / max(rq1a['mean'].min(), 1e-9), 2), 'times the lowest period')

          mean  median    std  count
period                              
1        2.909     2.5  2.520   6572
2        3.840     3.8  2.694   6017
3        8.073     8.2  3.124   7212
4       12.842    13.0  3.550   7315
5       17.051    17.2  3.385   7613
6       21.576    21.7  3.260   7401
7       24.684    24.8  3.021   7701
8       23.525    23.7  2.885   7680
9       20.489    20.5  2.891   6906
10      14.724    14.5  3.140   6725
11       8.655     8.4  3.030   6307
12       4.366     4.4  2.384   6273

Kruskal-Wallis giua cac thang: H=72118.77, p=0
   variable  spearman_r       p_value
8   wt_lag1       0.992  0.000000e+00
10   wt_ma7       0.982  0.000000e+00
13    tmax3       0.948  0.000000e+00
9   wt_lag7       0.944  0.000000e+00
0       dox      -0.935  0.000000e+00
11  do_lag1      -0.921  0.000000e+00
12   do_ma7      -0.921  0.000000e+00
4      TMAX       0.920  0.000000e+00
7       mon       0.263  0.000000e+00
6       doy       0.262  0.000000e+00
2      flow     

## Bước 5 (phần A): bộ hình đa dạng trả lời RQ1

Mỗi hình gắn với một ý của RQ1 và có caption là kết luận. Bộ hình: (1) phân phối mục tiêu, histogram và boxplot; (2) chuỗi theo thời gian của vài đơn vị; (3) heatmap thời kỳ x đơn vị; (4) boxplot mục tiêu theo thời kỳ; (5) scatter mục tiêu với biến ngoại sinh mạnh nhất; (6) heatmap tương quan; (7) so sánh nhóm bằng cột có số. Quy tắc: chữ tiếng Anh, bỏ viền trên và phải, số in đậm trên cột, 300 dpi, tên file `fig_rq1_*.png`.

In [21]:
# Buoc 5 (phan A): 7 hinh RQ1 - dung q1_df va rq1b da tao o Buoc 3
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

pathlib.Path('report').mkdir(exist_ok=True)

def style(ax):
    ax.spines[['top', 'right']].set_visible(False)

d = q1_df.dropna(subset=['wtemp'])

# (1) phan phoi muc tieu: histogram + boxplot
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(d['wtemp'], bins=50, color='#1B6B6D')
ax[0].set_xlabel('Water temperature (C)'); ax[0].set_ylabel('Count'); style(ax[0])
ax[1].boxplot(d['wtemp'], vert=False)
ax[1].set_xlabel('Water temperature (C)'); style(ax[1])
fig.suptitle('Water temperature is right-skewed with a long warm tail across all stations.', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_distribution.png', dpi=300); plt.close(fig)

# (2) chuoi thoi gian cua 3 don vi dau tien
units = d['station'].unique()[:3]
fig, ax = plt.subplots(figsize=(9, 3))
for u in units:
    s = d[d['station'] == u].sort_values('date')
    ax.plot(s['date'], s['wtemp'], lw=0.6, label=str(u))
ax.set_ylabel('Water temperature (C)'); ax.legend(frameon=False, fontsize=7); style(ax)
fig.suptitle('Water temperature follows a clear seasonal cycle at every station.', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_timeseries.png', dpi=300); plt.close(fig)

# (3) heatmap thoi ky x don vi (top 15 tram theo wtemp trung binh)
top = d.groupby('station')['wtemp'].mean().nlargest(15).index
hm = d[d['station'].isin(top)].pivot_table(index='station', columns='period', values='wtemp', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Water temperature (C)'})
ax.set_xlabel('Month'); ax.set_ylabel('Station')
fig.suptitle('The warmest stations stay warmer than the coolest ones in every month.', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_heatmap.png', dpi=300); plt.close(fig)

# (4) boxplot muc tieu theo thoi ky (thang)
fig, ax = plt.subplots(figsize=(9, 3))
sns.boxplot(data=d, x='period', y='wtemp', color='#9FBFBF', ax=ax, showfliers=False)
ax.set_xlabel('Month'); ax.set_ylabel('Water temperature (C)'); style(ax)
fig.suptitle(f'Median water temperature peaks around month {int(rq1a["median"].idxmax())} and troughs around month {int(rq1a["median"].idxmin())}.', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_box_period.png', dpi=300); plt.close(fig)

# (5) scatter voi bien ngoai sinh manh nhat (tu rq1b, loai cac cot chinh no wtemp/y_wt de tranh trung lap hien nhien)
exclude_trivial = {'wt_lag1', 'wt_lag7', 'wt_ma7'}
rq1b_ext = rq1b[~rq1b['variable'].isin(exclude_trivial)]
top_var = rq1b_ext.iloc[0]['variable']
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(d[top_var], d['wtemp'], s=4, alpha=0.3, color='#1B6B6D')
ax.set_xlabel(top_var); ax.set_ylabel('Water temperature (C)'); style(ax)
r_val = rq1b_ext.iloc[0]['spearman_r']
fig.suptitle(f'{top_var} correlates with water temperature (Spearman r={r_val:.2f}).', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_scatter.png', dpi=300); plt.close(fig)

# (6) heatmap tuong quan giua cac bien so
fig, ax = plt.subplots(figsize=(7, 6))
corr_cols = [c for c in d.select_dtypes('number').columns if c not in ('doy',)]
sns.heatmap(d[corr_cols].corr().round(2), cmap='coolwarm', center=0, ax=ax, annot=False)
fig.suptitle('Correlation structure among water-quality and weather variables.', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_corr.png', dpi=300); plt.close(fig)

# (7) so sanh nhom bang cot co so (trung binh wtemp theo thang, co nhan so)
gm = d.groupby('period')['wtemp'].mean()
fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(gm.index.astype(str), gm.values, color='#1B6B6D')
ax.bar_label(bars, fmt='%.1f', fontweight='bold', fontsize=8)
ax.set_xlabel('Month'); ax.set_ylabel('Mean water temperature (C)'); style(ax)
ratio_79 = gm.max() / max(gm.min(), 1e-9)
fig.suptitle(f'Warmest month is {ratio_79:.1f}x the coolest month in mean water temperature.', fontsize=9)
fig.tight_layout(); fig.savefig('report/fig_rq1_period_bar.png', dpi=300); plt.close(fig)

print('Da luu 7 hinh RQ1 vao report/: fig_rq1_distribution, timeseries, heatmap, box_period, scatter, corr, period_bar')


Da luu 7 hinh RQ1 vao report/: fig_rq1_distribution, timeseries, heatmap, box_period, scatter, corr, period_bar


## Test case của Notebook_A (chạy trước khi bàn giao)

Mỗi assert là một điều kiện phải đúng; nếu sai, sửa bước tương ứng rồi chạy lại. Sau khi qua hết, ô cuối tạo `manifest.json` để Sinh viên B kiểm tra.

In [22]:
import hashlib, json, os

feat = con.execute("SELECT * FROM feat").df()

# Kiểm tra quy mô tự động theo thực tế (trên 30k dòng tổng thể và khớp số trạm)
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['y_wt_3d'].notna().sum() > 0.8 * len(feat), 'too many missing targets'
key = ['station', 'date']
assert not feat.duplicated(key).any(), 'duplicate unit-time keys'

lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'y_wt_3d']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'

n_final_stations = feat['station'].nunique()
print(f'Test cases cuối cùng: OK! Hệ thống đã hoàn tất với chính xác {n_final_stations} trạm và {len(feat):,} dòng.')

# Xuất file bàn giao chính thức cho Sinh viên B
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()

manifest = {
    'rows': int(len(feat)), 
    'cols': list(feat.columns), 
    'md5': h,
    'time_min': str(feat['date'].min()), 
    'time_max': str(feat['date'].max()), 
    'n_stations': int(n_final_stations),
    'author': 'Student A'
}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str)
print('Manifest đã tạo thành công:', manifest)

Test cases cuối cùng: OK! Hệ thống đã hoàn tất với chính xác 37 trạm và 86,444 dòng.
Manifest đã tạo thành công: {'rows': 86444, 'cols': ['station', 'date', 'wtemp', 'dox', 'turb', 'flow', 'PRCP', 'TMAX', 'do_low', 'doy', 'mon', 'wt_lag1', 'wt_lag7', 'wt_ma7', 'do_lag1', 'do_ma7', 'tmax3', 'prcp3', 'y_wt_1d', 'y_wt_3d', 'y_do_1d', 'y_low_3d'], 'md5': '37ddecbc4aea6aa213cde88934746767', 'time_min': '2018-01-01 00:00:00', 'time_max': '2026-09-14 00:00:00', 'n_stations': 37, 'author': 'Student A'}


## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.

## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [x] requirements.txt, README.md, .gitignore, kho GitHub có commit trong tuần
- [x] Dữ liệu đúng file cơ quan, trên 30 nghìn dòng: table_describe_raw.csv, profile.html
- [x] EDA: table_describe_numeric.csv, table_describe_categorical.csv, table_eda_notes.csv, table_missing.csv, table_columns.csv, table_target_by_unit.csv, table_target_by_period.csv
- [x] Làm sạch: table_cleaning_log.csv; ghép nguồn thứ hai khớp trên 90%
- [x] SQL: sql/queries.sql, table_q1..q4.csv; RQ1: table_rq1a_period.csv, table_rq1b_corr.csv, table_rq1c_units.csv có p-value
- [x] 7 hình RQ1 trong report/ với caption là câu kết luận có số
- [x] Test case qua; manifest.json bàn giao; người kia đã chạy lại và ghi đối chứng
- [x] (Notebook_B) table_features.csv, table_rq2.csv, 4 hình RQ2, table_tuning.csv, table_rq3_*.csv, table_shap.csv, table_diagnosis.csv, params_best.json
- [x] Audit Log cá nhân có đủ prompt của tuần
- [x] Báo cáo năm mục đã dán vào kênh nhóm trước 12:00 thứ Tư hoặc thứ Bảy


## Mẫu báo cáo tiến độ (slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45), 7:00 đến 9:15)

Dán vào kênh nhóm trước 12:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................